# BBEATSx Phase 3: MCMC Diagnostics & Convergence Analysis

This notebook evaluates the MCMC sampling efficiency and convergence properties of BBEATSx, following the research plan §3.5.

We investigate:
1. **Multi-chain convergence**: Gelman-Rubin $\hat{R}$ diagnostic across independent chains.
2. **Mixing efficiency**: Effective Sample Size (ESS) and Autocorrelation Time for parameter traces (e.g., $\sigma^2$).
3. **Initialization Strategy**: Grow-from-Root (GFR) vs. Random Initialization mixing speed (XBART-style initialization benefit).

## Google Colab Setup

In [ ]:
# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running in Google Colab. Setting up workspace...")
    import os
    if not os.path.abspath(".").endswith("simulations"):
        if not os.path.exists("BBEATSx"):
            !git clone https://github.com/hugogobato/BBEATSx.git
        !pip install ./BBEATSx
        %cd BBEATSx/simulations
    else:
        print("Already in simulations directory.")

import sys
import os
sys.path.append(os.path.abspath("."))
sys.path.append(os.path.abspath(".."))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bbeatsx import BBEATSx, make_config
import utils

## 1. Multi-chain Convergence & Gelman-Rubin $\hat{R}$

We fit BBEATSx three times on DGP 1 with different random seeds to generate independent MCMC chains. We evaluate the convergence of $\sigma^2$ draws using the Gelman-Rubin statistic ($\hat{R}$), which should be close to 1.0 (typically $<1.05$) for converged chains.

In [ ]:
# Generate data
t, y, true_comps = utils.generate_dgp1(n=200, seed=42)

# Run 3 independent chains
num_chains = 3
chains_sigma2 = []

print("Running independent chains...")
for c in range(num_chains):
    # Configure model
    cfg = make_config(
        periods=[(12, 3)],
        lags=(1,),
        trend="spline",
        errors="homo",
        num_gfr=10,
        num_burnin=150,
        num_mcmc=300,
        seed=c * 100
    )
    
    model = BBEATSx(cfg).fit(y)
    chains_sigma2.append(model.sampler_.sigma2_draws_)

chains_sigma2 = np.array(chains_sigma2)
print(f"Chains shape: {chains_sigma2.shape} (num_chains, num_draws)")

In [ ]:
# Compute Gelman-Rubin R-hat
r_hat = utils.compute_gelman_rubin(chains_sigma2)
print(f"Gelman-Rubin R-hat for sigma^2: {r_hat:.4f}")

# Plot overlaid traces
plt.figure(figsize=(10, 4))
for c in range(num_chains):
    plt.plot(chains_sigma2[c], label=f"Chain {c+1}", alpha=0.7)
plt.xlabel("MCMC Draw Index (post-burnin)")
plt.ylabel("sigma^2")
plt.title(f"Trace Plot of sigma^2 across {num_chains} Chains (R-hat = {r_hat:.4f})")
plt.legend()
plt.grid(True)
plt.show()

## 2. Mixing Efficiency & Effective Sample Size (ESS)

We fit a single long chain (1000 MCMC draws after 300 burnin) to compute the Effective Sample Size (ESS) and analyze the autocorrelation function (ACF) of $\sigma^2$ and component draws.

In [ ]:
print("Running a long MCMC chain...")
cfg_long = make_config(
    periods=[(12, 3)],
    lags=(1,),
    trend="spline",
    errors="homo",
    num_gfr=10,
    num_burnin=300,
    num_mcmc=1000,
    seed=42
)

model_long = BBEATSx(cfg_long).fit(y)
draws_sigma2 = np.array(model_long.sampler_.sigma2_draws_)
print("Fit complete.")

In [ ]:
# Compute ESS
ess_sigma2 = utils.compute_ess(draws_sigma2)
ess_pct = (ess_sigma2 / len(draws_sigma2)) * 100
print(f"sigma^2 ESS: {ess_sigma2:.2f} / {len(draws_sigma2)} ({ess_pct:.1f}% efficiency)")

# Compute Autocorrelation
lags = np.arange(1, 50)
autocorrs = [utils.compute_autocorr(draws_sigma2, lag) for lag in lags]

plt.figure(figsize=(8, 4))
plt.bar(lags, autocorrs, width=0.6, edgecolor='black', alpha=0.7)
plt.axhline(0, color='black', linestyle='-')
plt.axhline(2 / np.sqrt(len(draws_sigma2)), color='blue', linestyle='--', label='95% Significance Band')
plt.axhline(-2 / np.sqrt(len(draws_sigma2)), color='blue', linestyle='--')
plt.xlabel("Lag")
plt.ylabel("Autocorrelation")
plt.title("Autocorrelation Function (ACF) for sigma^2")
plt.legend()
plt.show()

## 3. Initialization strategy: Grow-From-Root (GFR) vs. Random Initialization

We evaluate BBEATSx's warm-start strategy: **Grow-From-Root (GFR)**. Under standard BART, trees are initialized to small values or random splits. GFR builds trees sequentially using a boosting-like step (XBART strategy) prior to MCMC. This should yield instant convergence compared to starting from random splits. 

We run two samplers with zero burn-in: one with `num_gfr=15` and one with `num_gfr=0` (random init). We plot the trace of $\sigma^2$ from iteration 0 to 100.

In [ ]:
# We run the raw sampler directly or configure BBEATSx with num_burnin=0
print("Running GFR initialization comparison...")

# 1. With GFR Warm-start
cfg_gfr = make_config(
    periods=[(12, 3)],
    lags=(1,),
    trend="spline",
    errors="homo",
    num_gfr=15,
    num_burnin=0,
    num_mcmc=100,
    seed=123
)
model_gfr = BBEATSx(cfg_gfr).fit(y)
trace_gfr = np.array(model_gfr.sampler_.sigma2_draws_)

# 2. With Random Initialization (No GFR)
cfg_rand = make_config(
    periods=[(12, 3)],
    lags=(1,),
    trend="spline",
    errors="homo",
    num_gfr=0,
    num_burnin=0,
    num_mcmc=100,
    seed=123
)
model_rand = BBEATSx(cfg_rand).fit(y)
trace_rand = np.array(model_rand.sampler_.sigma2_draws_)

print("Done.")

In [ ]:
# Plot the two trajectories
plt.figure(figsize=(10, 5))
plt.plot(trace_gfr, label="With GFR Initialization (num_gfr=15)", color='blue', linewidth=2)
plt.plot(trace_rand, label="Random Initialization (num_gfr=0)", color='red', linewidth=1.5, linestyle='--')
plt.axhline(0.4**2, color='black', linestyle=':', label='True Noise Variance (DGP 1)')

plt.xlabel("MCMC Iteration")
plt.ylabel("sigma^2 Draw")
plt.title("Trace Convergence: GFR vs. Random Initialization (No Burn-in)")
plt.legend()
plt.grid(True)
plt.show()